In [ ]:
 #Install Required Dependencies
%pip install --upgrade pip

# Uninstall conflicting packages
%pip uninstall -y langchain_classic langchain-core langchain-openai langchain-community langchain langchain-chroma chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25 weaviate-client ragas wikipedia langchain-weaviate langchain-together langchain-experimental tiktoken langgraph langchain-tavily

# PRE-STEP: Install Required Dependencies
%pip install langchain==1.1.0
%pip install langgraph==1.0.4
%pip install langchain-openai==1.1.0
%pip install langchain-chroma==1.0.0
%pip install chromadb==1.3.5
%pip install python-dotenv==1.1.1
%pip install pydantic==2.12.3

In [ ]:
# Cell 1: Setup and load the trained agent state from Lab 18-1
import os
import json
import pickle
from datetime import datetime
import pandas as pd
import numpy as np
from typing import Dict, List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from dotenv import load_dotenv

# Import the new architecture components
from coala_agent import CoALAAgent
from domain_investment.investment_advisor_agent import InvestmentAdvisorAgent
from domain_investment.investment_advisor_prompts import (
    PROMPT_MEMORY_OPTIMIZATION, GRADIENT_CRITIQUE, GRADIENT_PROPOSAL,
    METAPROMPT_SURFACE, METAPROMPT_DEEP, METAPROMPT_SYNTHESIS
)
from domain_investment.investor_test_scenarios import (
    run_prompt_memory_test,
    run_gradient_test,
    test_agents_with_queries,
    test_response_consistency,
    compare_agent_performance
)


from domain_agent import DomainProcedure

load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [ ]:
# Cell 2: Define function to save checkpoints
#  Create checkpoint of current state
def save_checkpoint(agent: CoALAAgent, checkpoint_name: str):
    """Save agent state for comparison baseline"""
    # Get procedural memory stats
    proc_stats = agent.procedural_memory.get_stats() if hasattr(agent, 'procedural_memory') else {}
    
    checkpoint = {
        'timestamp': datetime.now().isoformat(),
        'episodic_count': len(agent.vector_store.get()["ids"]) if hasattr(agent.vector_store, 'get') else 0,
        'procedural_stats': proc_stats,
        'learned_strategies': list(agent.procedural_memory.global_procedures.keys()) if hasattr(agent, 'procedural_memory') else [],
        'current_performance': proc_stats.get('avg_success_rate', 0)
    }
    
    with open(f"checkpoints/{checkpoint_name}.pkl", "wb") as f:
        pickle.dump(checkpoint, f)
    
    return checkpoint

# Create checkpoint directory
os.makedirs("checkpoints", exist_ok=True)

print("🔄 Loading trained agent from Lab 18-1...")

# Create domain agent
domain_agent = InvestmentAdvisorAgent()
